# 第 7 章：測試驅動開發 —— 從 assert 到 pytest

> 🎬 **情境**
> 阿宏的系統已經有 12 個函式，你累積了 40 幾個 `assert`。
> 今天他說：「會員折扣改成滿 200 才有。」
>
> 你改完，從頭跑一遍 —— 第 7 個掛了。修好，再從頭跑 —— 第 23 個掛了。
> 跑到第 23 個的時候，前面 22 個又被改壞了嗎？你不知道，
> 因為 **`assert` 一失敗，後面的就不會執行了**。

## 7.1 😩 土法煉鋼：一整片 assert 的極限

In [ ]:
MENU = {"珍珠奶茶": {"中杯": 55, "大杯": 65}, "冬瓜檸檬": {"中杯": 40, "大杯": 45}}


def price_of(name, size, is_member=False):
    price = MENU[name][size]
    return round(price * 0.9) if is_member else price


try:
    assert price_of("珍珠奶茶", "大杯") == 65
    assert price_of("珍珠奶茶", "中杯") == 66       # ← 故意寫錯，看會怎樣
    assert price_of("冬瓜檸檬", "大杯") == 45       # ← 以下全部不會執行
    assert price_of("冬瓜檸檬", "中杯") == 40
except AssertionError:
    print("🔴 第 2 行失敗，後面兩行根本沒跑。")
    print("   而且它沒告訴你『預期 66、實際 55』。")

`assert` 到了這個規模，有四個問題：

| 問題 | 具體狀況 |
| --- | --- |
| **失敗就中斷** | 一次只能知道一個錯，修 40 個要跑 40 次 |
| **沒有報表** | 不知道總共幾個、過了幾個 |
| **不知道為什麼錯** | 只說「不成立」，沒說「預期 55、實際 60」 |
| **沒有組織** | 40 個混在一起，分不出哪些測折扣、哪些測菜單 |

你需要的很明確：**跑完全部、告訴我哪些過哪些沒過、失敗時告訴我預期和實際。**

## 7.2 TDD 的完整循環

```
     🔴 紅：寫一個會失敗的測試
           ↓
     🟢 綠：寫剛好夠通過的程式
           ↓
     🔵 重構：在測試保護下整理程式
           ↓
        回到 🔴
```

**🔴 紅燈紀律**：一次只寫一個測試；執行它，確認它是因為「功能還沒做」而失敗，
不是因為測試本身有 typo；看清楚錯誤訊息長什麼樣。

**🟢 綠燈紀律**：寫**剛好夠**通過的程式，不要多寫。
如果測試只要求 `price_of("珍珠奶茶","大杯") == 65`，你寫 `return 65` 是完全合法的第一步 ——
它逼你用下一個測試說清楚真正的規則（這招叫 *fake it till you make it*）。

**🔵 重構紀律**：不能新增功能，只能改善結構；每改一小步就跑一次測試；變紅立刻退回。

### 為什麼順序不能顛倒？

**先寫程式再補測試**，你的測試會被實作綁架 —— 你只會測「你寫得出來的東西」，
而不是「使用者真正需要的東西」。

In [ ]:
# 先實作的人寫出這個，然後補一個測試
def 計算折扣(金額, 折數):
    return 金額 * 折數


print("先實作的人只會驗證他寫的那行：", 計算折扣(100, 0.9) == 90.0)

# 先寫測試的人會先問「使用者會怎麼用？」，於是問出一堆問題：
print("\n先寫測試的人會問：")
print("  計算折扣(100, 0.9) 要回 90 還是 90.0？")
print("  計算折扣(65, 0.9) = 58.5，要進位嗎？→", 計算折扣(65, 0.9))
print("  計算折扣(100, 1.5) 折數大於 1 合理嗎？→", 計算折扣(100, 1.5), "← 竟然越打越貴")
print("\n第二組問題會逼出一個好上許多的函式。")

## 7.3 三個測試工具，循序漸進

| 工具 | 需要安裝 | 適合 | 特色 |
| --- | --- | --- | --- |
| `assert` | ❌ | 學習、Notebook、臨時檢查 | 零成本 |
| `doctest` | ❌ | 函式的使用範例 | **說明文件就是測試** |
| `unittest` | ❌ | 不能裝套件的環境 | 標準庫，完整但囉嗦 |
| `pytest` | ✅ | **實際專案** | 業界標準，最簡潔 |

## 7.4 doctest：讓說明文件變成測試

In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def round_half_up(value: float) -> int:
    """台灣習慣的四捨五入（逢五進一）。

    >>> round_half_up(58.5)
    59
    >>> round_half_up(2.5)
    3
    >>> round_half_up(204.75)
    205
    """
    return int(Decimal(str(value)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


import doctest

doctest.run_docstring_examples(round_half_up, {"round_half_up": round_half_up}, verbose=True)

In [ ]:
# 故意寫錯一個，看 doctest 怎麼報錯
def 有問題的函式(x):
    """把數字加倍。

    >>> 有問題的函式(3)
    7
    """
    return x * 2


print("👇 doctest 會告訴你「預期什麼、實際什麼」：")
doctest.run_docstring_examples(有問題的函式, {"有問題的函式": 有問題的函式}, verbose=False)

**doctest 的殺手級價值：文件和程式不會再脫節。**
你改了行為卻忘了改文件？doctest 會失敗，逼你更新。

在 `.py` 檔裡執行全部 doctest：

```bash
python3 -m doctest 你的檔案.py -v
```

⚠️ **限制**：doctest 比對的是**輸出的字串**，一字不差。所以：

- 浮點數 `0.30000000000000004` 這種東西不適合寫進 doctest
- 例外訊息包含路徑時會失敗

**準則：doctest 寫「給人看的漂亮範例」，複雜的邊界測試交給 pytest。**

In [ ]:
# 親眼看看為什麼浮點數不適合 doctest
print(0.1 + 0.2)          # 你以為是 0.3，doctest 會因為這個差異而失敗

## 7.5 unittest：標準庫的測試框架

不需要安裝任何東西，**在任何作業系統、任何 Python 環境都能用**。

In [ ]:
import unittest


class TestPricing(unittest.TestCase):
    def setUp(self):
        """每個測試方法執行前都會跑一次，用來準備資料。"""
        self.menu = {"珍珠奶茶": {"中杯": 55, "大杯": 65}}

    def test_一般價格(self):
        self.assertEqual(price_of("珍珠奶茶", "大杯"), 65)

    def test_會員折扣(self):
        self.assertEqual(price_of("珍珠奶茶", "大杯", is_member=True), 59)

    def test_不存在的品項會拋出_KeyError(self):
        with self.assertRaises(KeyError):
            price_of("綠茶", "大杯")


# 在 Jupyter 裡要這樣跑（argv=[""] 和 exit=False 缺一不可）
unittest.main(argv=[""], exit=False, verbosity=2)

看到了嗎？

- **全部跑完**（不會因為一個失敗就中斷）
- **有統計**（Ran 3 tests，FAILED failures=1）
- **有預期/實際對照**（`AssertionError: 58 != 59`）

三個 `assert` 做不到的事，它都做到了。

### 常用斷言方法

| 方法 | 意思 |
| --- | --- |
| `assertEqual(a, b)` | a == b |
| `assertTrue(x)` / `assertFalse(x)` | 真假 |
| `assertIn(a, b)` | a in b |
| `assertIsNone(x)` | x is None |
| `assertRaises(錯誤型別)` | **應該要拋出例外** |
| `assertAlmostEqual(a, b)` | 浮點數比較 |

In [ ]:
# 把上面那個失敗的測試修好（price_of 用 round 會得到 58，要用 round_half_up）
def price_of(name, size, is_member=False):
    price = MENU[name][size]
    return round_half_up(price * 0.9) if is_member else price


unittest.main(argv=[""], exit=False, verbosity=2)

🟢 三個全過。**注意我們沒有改任何一行測試，只改了實作** —— 這就是測試的意義。

## 7.6 pytest：業界標準

```bash
pip install pytest
```

同樣的測試，pytest 版本 —— **沒有類別、沒有繼承、就用原生的 `assert`**：

```python
# tests/test_pricing.py
import pytest
from drinkshop.pricing import price_of

def test_一般價格():
    assert price_of("珍珠奶茶", "大杯") == 65

def test_不存在的品項會拋出_KeyError():
    with pytest.raises(KeyError):
        price_of("綠茶", "大杯")
```

失敗時 pytest 自動幫你拆解 `assert`：

```
E       assert 60 == 59
E        +  where 60 = price_of('珍珠奶茶', '大杯', is_member=True)
```

它連「這個 60 是從哪個呼叫來的」都告訴你了。
這叫 *assertion rewriting*，是 pytest 最受歡迎的功能。

In [ ]:
# 檢查這台電腦有沒有裝 pytest
try:
    import pytest

    print("🟢 pytest", pytest.__version__, "已安裝，下面的範例可以直接跑")
    有pytest = True
except ImportError:
    print("⚪ 沒有安裝 pytest。用 pip install pytest 安裝，")
    print("   或繼續用上面的 unittest —— 觀念完全一樣。")
    有pytest = False

### pytest 的三個必學功能

**1️⃣ 參數化：一個測試涵蓋很多案例**

```python
@pytest.mark.parametrize("name,size,expected", [
    ("珍珠奶茶", "中杯", 55),
    ("珍珠奶茶", "大杯", 65),
    ("冬瓜檸檬", "中杯", 40),
    ("冬瓜檸檬", "大杯", 45),
])
def test_菜單價格(name, size, expected):
    assert price_of(name, size) == expected
```

這會產生 **4 個獨立的測試**，任何一個失敗不影響其他三個，報表會告訴你是哪一組。

**2️⃣ fixture：共用的準備工作**

```python
@pytest.fixture
def 今日訂單():
    return [65, 45, 120]

def test_總營收(今日訂單):        # 參數名對上 fixture 名，pytest 自動注入
    assert sum(今日訂單) == 230
```

**3️⃣ `tmp_path`：跨平台的暫存目錄**

```python
def test_存檔(tmp_path):
    檔案 = tmp_path / "報表.csv"      # 自動建立、測試結束自動刪除
    存檔(檔案, "內容")
    assert 檔案.read_text(encoding="utf-8") == "內容"
```

⚠️ **永遠不要在測試裡寫死 `/tmp/test.csv` 或 `C:\temp\test.csv`** —— 換個作業系統就爆。
**`tmp_path` 是跨平台的正確答案**（第 9 章深入）。

In [ ]:
# unittest 也有等價的參數化做法（subTest），不用裝任何東西
class TestMenu(unittest.TestCase):
    def test_菜單價格(self):
        案例 = [
            ("珍珠奶茶", "中杯", 55),
            ("珍珠奶茶", "大杯", 65),
            ("冬瓜檸檬", "中杯", 40),
            ("冬瓜檸檬", "大杯", 45),
        ]
        for name, size, expected in 案例:
            with self.subTest(name=name, size=size):     # 每個案例獨立回報
                self.assertEqual(price_of(name, size), expected)


unittest.main(argv=[""], exit=False, verbosity=2)

### pytest 常用指令

```bash
pytest                    # 跑全部
pytest -v                 # 顯示每個測試的名字
pytest -x                 # 第一個失敗就停
pytest -k "會員"          # 只跑名字含「會員」的
pytest tests/test_pricing.py::test_會員折扣   # 只跑指定一個
pytest --lf               # 只重跑上次失敗的 ← 超好用
```

## 7.7 怎麼寫出好測試

### AAA 結構：Arrange / Act / Assert

In [ ]:
def test_會員買大杯珍奶有九折():
    # Arrange 準備
    name, size = "珍珠奶茶", "大杯"

    # Act 執行
    實際 = price_of(name, size, is_member=True)

    # Assert 驗證
    assert 實際 == 59


test_會員買大杯珍奶有九折()
print("🟢 三段式讓測試好讀：給了什麼、做了什麼、期待什麼")

### 測試名稱要說「行為」，不是「函式名」

```python
def test_price_of():                 # ✖ 看不出在測什麼
def test_會員買大杯有九折():           # ✔ 這根本就是一句規格
def test_品項不存在時拋出KeyError():   # ✔
```

**好的測試名稱集合起來，就是一份永遠不會過期的規格書。**

### 一定要測邊界

**bug 最愛藏在邊界。** 每個條件式都問自己：

| 條件 | 要測的值 |
| --- | --- |
| `金額 >= 200` 打折 | 199、**200**、201 |
| 清單處理 | **空清單**、一個元素、很多元素 |
| 字串處理 | **空字串**、只有空白 |
| 除法 | **除數為 0** |
| 數量 | 0、負數 |

In [ ]:
def 平均客單價(訂單):
    return sum(訂單) / len(訂單)


print("一般情況：", 平均客單價([100, 200]))

try:
    平均客單價([])            # ← 最容易被忘記的邊界
except ZeroDivisionError as 錯誤:
    print("🔴 空清單直接炸掉：", type(錯誤).__name__)
    print("   打烊時如果一筆訂單都沒有，你的日報表就掛了。")

### 什麼**不該**測

別被「覆蓋率要 100%」綁架。以下不值得測：

- **第三方套件的行為**（pandas 的 sum 對不對不是你的責任）
- **Python 內建功能**（不用測 `len()` 會不會算錯）
- **純粹的資料定義**（`MENU = {...}` 沒有邏輯可測）
- **只把參數轉手傳給別人的一行函式**

**測「你自己寫的邏輯」和「會出錯的地方」。**

### 測試要快

慢的測試沒人會跑，沒人跑的測試等於沒有。單元測試應該在**幾毫秒**內完成；
不要在單元測試裡連資料庫、打 API、`sleep()`。

## 7.8 🔴🟢🔵 完整實作一輪

阿宏的新需求：**「滿 200 才給會員折扣。」**

### 🔴 第一圈：紅

In [ ]:
def test_未滿200的會員不打折():
    assert 結帳(金額=150, 是會員=True) == 150


try:
    test_未滿200的會員不打折()
except NameError as 錯誤:
    print("🔴", 錯誤)

### 🟢 第一圈：綠（寫剛好夠的）

In [ ]:
def 結帳(金額, 是會員):
    return 金額


test_未滿200的會員不打折()
print("🟢 通過")
print("是的，這個實作完全沒用到「是會員」—— 這很正常。")
print("現在的測試只要求這件事，下一個測試會逼我寫出真正的邏輯。")

### 🔴 第二圈：紅

In [ ]:
def test_滿200的會員打九折():
    assert 結帳(金額=200, 是會員=True) == 180


try:
    test_滿200的會員打九折()
except AssertionError:
    print("🔴 assert 200 == 180 失敗 —— 正是我要的紅燈")

### 🟢 第二圈：綠

In [ ]:
def 結帳(金額, 是會員):
    if 是會員 and 金額 >= 200:
        return round_half_up(金額 * 0.9)
    return 金額


test_未滿200的會員不打折()
test_滿200的會員打九折()
print("🟢 兩個都過")

### 🔴 第三圈：補邊界

In [ ]:
def test_剛好199不打折():
    assert 結帳(金額=199, 是會員=True) == 199


def test_非會員滿200也不打折():
    assert 結帳(金額=200, 是會員=False) == 200


test_剛好199不打折()
test_非會員滿200也不打折()
print("🟢 直接綠。這也是有價值的 —— 它們變成了「規則的護欄」，")
print("   以後有人改壞這個行為，它們會立刻叫出來。")

### 🔵 重構

In [ ]:
MEMBER_THRESHOLD = 200       # 把魔術數字命名
MEMBER_RATE = 0.9


def 結帳(金額: int, 是會員: bool) -> int:
    """回傳應收金額；會員消費滿 MEMBER_THRESHOLD 元享 9 折。"""
    if 是會員 and 金額 >= MEMBER_THRESHOLD:
        return round_half_up(金額 * MEMBER_RATE)
    return 金額


for 測試 in [test_未滿200的會員不打折, test_滿200的會員打九折,
            test_剛好199不打折, test_非會員滿200也不打折]:
    測試()
    print("🟢", 測試.__name__)

print("\n重構成功。隔天阿宏說「門檻改成 300」，你只要改一個常數，而且立刻知道有沒有改壞。")

## 📌 本章速記

```python
# 三步驟
🔴 寫會失敗的測試 → 🟢 寫剛好夠通過的程式 → 🔵 重構 → 回到 🔴

# doctest：文件即測試
""" >>> f(2)
    4 """
import doctest; doctest.testmod()

# unittest：標準庫，免安裝
class TestX(unittest.TestCase):
    def test_行為描述(self):
        self.assertEqual(實際, 預期)
        with self.assertRaises(ValueError): ...
        with self.subTest(x=x): ...          # 參數化
unittest.main(argv=[""], exit=False)         # Jupyter 裡這樣跑

# pytest：業界標準
def test_行為描述(): assert 實際 == 預期
with pytest.raises(ValueError): ...
@pytest.mark.parametrize("a,b,expected", [...])
def test_檔案(tmp_path): ...                 # 跨平台暫存目錄
pytest -v / -x / -k "關鍵字" / --lf

# 好測試的準則
AAA 結構、名稱描述行為、一個測試一件事、一定要測邊界、
不測第三方與內建、測試要快
```

---
## 🧪 練習

### 練習 7-1：用 TDD 做「提袋數量」
規則：一袋裝 6 杯，不滿一袋也要一個袋子，**0 杯要 0 個袋子**。

請先讀下面的測試（它們已經幫你把邊界想好了），再實作 `袋子數`。

In [ ]:
def 袋子數(杯數):
    pass  # 👉 你的實作


def test_零杯不需要袋子():
    assert 袋子數(0) == 0


def test_一杯要一個袋子():
    assert 袋子數(1) == 1


def test_剛好六杯要一個袋子():
    assert 袋子數(6) == 1


def test_七杯要兩個袋子():
    assert 袋子數(7) == 2


失敗 = []
for 測試 in [test_零杯不需要袋子, test_一杯要一個袋子, test_剛好六杯要一個袋子, test_七杯要兩個袋子]:
    try:
        測試()
        print("🟢", 測試.__name__)
    except AssertionError:
        失敗.append(測試.__name__)
        print("🔴", 測試.__name__)
print("\n全部通過！" if not 失敗 else f"\n還有 {len(失敗)} 個要修")

### 練習 7-2：補完 doctest
把 `???` 換成正確答案，讓 doctest 通過。

In [ ]:
def 折扣後金額(金額, 折數):
    """計算折扣後金額（四捨五入到整數）。

    >>> 折扣後金額(100, 0.9)
    90
    >>> 折扣後金額(65, 0.9)
    59
    """
    return round_half_up(金額 * 折數)


# 只檢查這一個函式的 doctest（在 .py 檔裡可以直接用 doctest.testmod() 檢查整個檔案）
執行器 = doctest.DocTestRunner(verbose=False)
for 測試 in doctest.DocTestFinder().find(折扣後金額):
    執行器.run(測試)

統計 = 執行器.summarize(verbose=False)
print("doctest：試了", 統計.attempted, "個，失敗", 統計.failed, "個")
print("🟢 通過" if 統計.failed == 0 else "🔴 有失敗")

### 練習 7-3：這個測試有什麼問題？

```python
def test_全部():
    assert price_of("珍珠奶茶", "大杯") == 65
    assert price_of("珍珠奶茶", "中杯") == 55
    assert 結帳(200, True) == 180
    assert 總營收([]) == 0
```

👉 提示：想想第二行失敗時，你能知道第三、四行的狀況嗎？測試名稱說得出它在測什麼嗎？

### 練習 7-4：找出遺漏的測試

這個函式已經有兩個測試，但還有一個嚴重的 bug 沒被測出來。**少了什麼測試？**

In [ ]:
def 平均客單價(訂單):
    return sum(訂單) / len(訂單)


def test_一般情況():
    assert 平均客單價([100, 200]) == 150


def test_單筆():
    assert 平均客單價([100]) == 100


test_一般情況()
test_單筆()
print("🟢 兩個測試都過了，看起來很安全…")
print("👉 但是 平均客單價([]) 呢？試著寫出那個測試，然後修好函式。")

---
📁 **完整範例**：`examples/` 有一整套 pytest 測試可以參考

➡️ 下一章：`08-模組與套件.ipynb` —— 檔案長到 400 行，該拆了。